# 08 Lintasan Refinement CARE KM


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import hashlib

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# Find paper root from the standard Drive path or from the local package.
_candidates = [
    Path(os.environ.get('CARE_KM_PAPER_ROOT', '')) if os.environ.get('CARE_KM_PAPER_ROOT') else None,
    _ROOT_REVISED,
    _ROOT_REVISED,
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
]
PAPER_ROOT = next((p.resolve() for p in _candidates if p is not None and (p/'LIB').is_dir() and (p/'DATA').is_dir()), None)
if PAPER_ROOT is None:
    raise FileNotFoundError('Root REVISED tidak ditemukan.')
LIB_DIR = PAPER_ROOT / 'LIB'
DATA_DIR = PAPER_ROOT / 'DATA'
OUT_DIR = stage_dir('08_Lintasan_Refinement_CARE_KM')
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))

import experiment_lib as el
import care_km as ck

CARE_CONFIG = dict(el.FROZEN_CARE_CONFIG)
K = el.K
METHODS = list(el.METHODS)
print('PAPER_ROOT:', PAPER_ROOT)
print('CARE config:', CARE_CONFIG)
print('CARE metadata:', el.frozen_care_metadata())


df, X_all, FEATURES = el.load_lms_dataset(PAPER_ROOT)
REPS = 20
FRACTION = 0.80
BASE_SEED = 5161600

CHECKPOINT = artifact_path('05_refinement_trajectory_raw.csv','08_Lintasan_Refinement_CARE_KM')
LEGACY_ALIAS = artifact_path('05_ablation_raw.csv','08_Lintasan_Refinement_CARE_KM')
SUMMARY_PATH = artifact_path('05_refinement_trajectory_summary.csv','08_Lintasan_Refinement_CARE_KM')
SUMMARY_ALIAS = artifact_path('05_ablation_summary.csv','08_Lintasan_Refinement_CARE_KM')

q = int(CARE_CONFIG['candidate_budget'])
derived_budget = ck.derive_refinement_budget(q)

if CHECKPOINT.exists():
    raw = pd.read_csv(CHECKPOINT)
else:
    raw = pd.DataFrame()

for rep in range(REPS):
    rng = np.random.default_rng(BASE_SEED + rep)
    idx = np.sort(
        rng.choice(
            len(X_all),
            size=int(len(X_all) * FRACTION),
            replace=False,
        )
    )
    X = X_all[idx]
    sample_hash = el.sample_index_hash(idx)
    record_resample('08_Lintasan_Refinement_CARE_KM','LMS',rep,idx,sample_hash)

    mask = (
        pd.to_numeric(raw.get('rep', pd.Series(dtype=float)), errors='coerce').eq(rep)
        if len(raw) else pd.Series(dtype=bool)
    )
    already_complete = False
    if len(raw) and mask.any():
        rrep = raw.loc[mask]
        already_complete = bool(
            rrep.get('is_final_configuration', pd.Series(dtype=bool))
                .astype(str).str.lower().eq('true').any()
        )
    if already_complete:
        continue

    print(f'rep={rep+1}/{REPS}: build one CARE-KM trajectory')

    _, trace, path = ck.care_init(
        X,
        K,
        return_trace=True,
        return_path=True,
        **CARE_CONFIG,
    )

    accepted = len(trace)
    stopping_reason = (
        'NO_POSITIVE_GAIN'
        if accepted < derived_budget
        else 'DERIVED_REFINEMENT_BUDGET_REACHED'
    )

    # Diagnostic states from the SAME trajectory.
    requested = [0, 1, 2, accepted]
    steps = []
    for step in requested:
        if step <= accepted and step not in steps:
            steps.append(step)

    for step in steps:
        init_modes = path[step]
        result = ck.kmodes_fit(X, init_modes, max_iter=el.MAX_ITER)
        row = ck.evaluate_clustering(
            X,
            None,
            result,
            init_modes,
            si_sample_size=el.SI_SAMPLE_SIZE,
            si_seed=BASE_SEED + rep,
        )

        if step == 0:
            configuration = 'Cao parent'
            method = 'Cao'
        elif step == accepted:
            configuration = 'CARE-KM bounded refinement'
            method = 'CARE-KM'
        else:
            configuration = f'CARE-KM correction {step}'
            method = 'CARE-KM diagnostic'

        row.update({
            'method': method,
            'rep': rep,
            'configuration': configuration,
            'refinement_step': step,
            'accepted_corrections': step,
            'candidate_budget': q,
            'derived_refinement_budget': derived_budget,
            'trajectory_accepted_exchanges': accepted,
            'stopping_reason': stopping_reason,
            'is_final_configuration': bool(step == accepted),
            'sample_hash': sample_hash,
        })
        raw = pd.concat([raw, pd.DataFrame([row])], ignore_index=True)

    el.save_csv(raw, CHECKPOINT)
    # Compatibility alias for downstream notebooks.
    el.save_csv(raw, LEGACY_ALIAS)

# Ringkasan selalu memakai checkpoint CSV tersimpan (6 desimal),
# sehingga first run dan resume memakai presisi yang sama dengan checkpoint CSV tersimpan.
raw = pd.read_csv(CHECKPOINT)

if raw['rep'].nunique() != REPS:
    raise RuntimeError('Refinement trajectory checkpoint belum lengkap 20 repetitions.')
if not (raw.groupby('rep')['sample_hash'].nunique() == 1).all():
    raise RuntimeError('Matched-sample integrity gagal.')

summary = (
    raw.groupby(
        [
            'configuration',
            'refinement_step',
            'candidate_budget',
            'derived_refinement_budget',
        ],
        dropna=False,
    )
    .agg(
        n=('rep','size'),
        initial_cost_mean=('initial_cost','mean'),
        initial_cost_sd=('initial_cost','std'),
        final_cost_mean=('final_cost','mean'),
        final_cost_sd=('final_cost','std'),
        dbi_mean=('dbi','mean'),
        dbi_sd=('dbi','std'),
        si_mean=('si','mean'),
        si_sd=('si','std'),
        iterations_mean=('iterations','mean'),
    )
    .reset_index()
    .sort_values('refinement_step')
)

el.save_csv(summary, SUMMARY_PATH)
el.save_csv(summary, SUMMARY_ALIAS)

display(summary)
print('Derived refinement budget:', derived_budget)
print('Intermediate states are diagnostic checkpoints, not hyperparameter variants.')
print('Refinement trajectory matched-resampling integrity: PASS')
